# 91 — Extract empirical source waveforms from position-coded nodal shot gathers

This notebook estimates an **empirical near-source waveform** from SmartSolo/nodal shot-gather events.

This version assumes the derived SDS / event products use **5-character station codes equal to receiver position in centimetres**:

```text
station = 08850  -> receiver x = 88.50 m
station = 10250  -> receiver x = 102.50 m
station = 12005  -> receiver x = 120.05 m
```

That means geometry can be recovered directly from the SEED station code, without needing to join against SmartSolo serial numbers.

It can run in two modes:

1. **Single-event mode**: load one event MiniSEED file and extract source waveforms.
2. **Batch mode**: loop over many event MiniSEED files in `events_mseed/`.

For each event, the notebook will:

- read the event waveform window,
- derive node geometry from position-coded station names,
- estimate the likely shot/source position along the line,
- select the closest one or two nodes,
- extract short source-waveform windows on each available component,
- compute spectra and frequency summary metrics,
- export files suitable for later SPECFEM2D source injection.

Important caveat: this is not a true deconvolved source-time function. It is an **empirical near-source waveform**:

```text
observed waveform ≈ source pulse * near-field propagation * ground coupling * receiver response
```

For modelling, that may still be useful if clearly documented.


## 1. Imports and configuration

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import read, UTCDateTime, Stream, Trace
from obspy.signal.trigger import classic_sta_lta, trigger_onset

warnings.filterwarnings("ignore", category=UserWarning)

# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")

# Root folder containing output from the detection/picking notebook.
# This is searched recursively for event MiniSEED files.
EVENT_MSEED_ROOT = PROJECT_ROOT / "nodal_shotgathers_position_codes_timewindows"

# Geometry / metadata table. Update path/sheet/column handling below as needed.
GEOMETRY_XLSX = PROJECT_ROOT / "metadata" / "glenn_smartsolo_nodal_metadata_with_estimated_coords.xlsx"
GEOMETRY_SHEET = None  # None = read all sheets if needed

# Output folder for source waveforms.
OUTDIR = PROJECT_ROOT / "nodal_source_waveforms_for_specfem2d"
OUTDIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# Run mode
# ---------------------------------------------------------------------
RUN_MODE = "batch"  # "single" or "batch"
SINGLE_EVENT_MSEED = None

# Moving trigger/source nodes should not be treated as fixed receiver positions.
# Include both the full SmartSolo serial form and any shortened derived station form.
MOVING_TRIGGER_STATIONS = {"45012806", "12806"}
# SINGLE_EVENT_MSEED = Path("/Volumes/tachyon/LBSSP_DATA/.../events_mseed/event_000001.mseed")

BATCH_SEARCH_ROOT = EVENT_MSEED_ROOT
EVENT_FILE_GLOB = "*.mseed"
EVENT_FILE_GLOB = "events_mseed/*.mseed"

# ---------------------------------------------------------------------
# Waveform extraction parameters
# ---------------------------------------------------------------------
BANDPASS_FREQMIN = 16.0
BANDPASS_FREQMAX = 128.0
STA_S = 0.005
LTA_S = 0.050
TRIGGER_ON = 3.5
TRIGGER_OFF = 1.2

SOURCE_GRID_DX_M = 0.5
VELOCITY_GRID_MPS = np.arange(80.0, 1200.0 + 1, 20.0)
MIN_PICK_COUNT_FOR_FIT = 4

SOURCE_PRE_S = 0.030
SOURCE_POST_S = 0.500
SPECTRUM_TAPER_FRACTION = 0.05
NORMALIZE_EXPORT = True
N_NEAREST_NODES = 2

print("OUTDIR:", OUTDIR)


OUTDIR: /Volumes/tachyon/LBSSP_DATA/nodal_source_waveforms_for_specfem2d


## 2. Geometry from position-coded station names

For the derived archive, the station code is the receiver position in centimetres.

Examples:

```text
08850 -> 88.50 m
10250 -> 102.50 m
12005 -> 120.05 m
```

The geometry table is therefore built from the stations actually present in each event file. Moving trigger/source nodes such as `12806` are excluded from geometry-based source-location fitting.


In [2]:
def station_code_to_x_m(station: str) -> float:
    """Convert 5-character position-coded station name to x in metres."""
    s = str(station).strip()
    if s in MOVING_TRIGGER_STATIONS:
        raise ValueError(f"{s} is marked as a moving trigger/source station")
    if not s.isdigit():
        raise ValueError(f"Station code is not numeric: {s}")
    if len(s) > 5:
        # Usually not a derived position code; probably a raw serial-like code.
        raise ValueError(f"Station code is too long for position-cm code: {s}")
    return int(s) / 100.0


def is_fixed_position_station(station: str) -> bool:
    s = str(station).strip()
    if s in MOVING_TRIGGER_STATIONS:
        return False
    if not s.isdigit():
        return False
    if len(s) > 5:
        return False
    return True


def geometry_from_stream_station_codes(st: Stream) -> pd.DataFrame:
    """Build geometry dataframe from position-coded station names in a Stream."""
    rows = []
    for sta in sorted(set(str(tr.stats.station).strip() for tr in st)):
        if not is_fixed_position_station(sta):
            continue
        try:
            x_m = station_code_to_x_m(sta)
        except Exception:
            continue
        rows.append({
            "station": sta,
            "x_m": x_m,
        })

    geom = pd.DataFrame(rows)
    if len(geom):
        geom = geom.drop_duplicates(subset=["station"]).sort_values("x_m").reset_index(drop=True)
    return geom


# Global geometry is no longer loaded from Excel. It is built per event.
geometry = pd.DataFrame(columns=["station", "x_m"])
print("Geometry will be derived from position-coded station names in each event.")


Geometry will be derived from position-coded station names in each event.


## 3. Arrival-picking and source-location helper functions

In [3]:
def stream_station_names(st: Stream) -> list[str]:
    return sorted(set(tr.stats.station for tr in st))


def robust_trace_energy(tr: Trace) -> float:
    x = np.asarray(tr.data, dtype=float)
    if x.size == 0:
        return np.nan
    return float(np.sqrt(np.nanmean(x**2)))


def preprocess_for_arrivals(st: Stream) -> Stream:
    st2 = st.copy()
    st2.detrend("demean")
    st2.detrend("linear")
    st2.taper(max_percentage=0.02)
    try:
        st2.filter("bandpass", freqmin=BANDPASS_FREQMIN, freqmax=BANDPASS_FREQMAX, corners=4, zerophase=True)
    except Exception as e:
        print(f"Bandpass failed: {e}")
    return st2


def choose_detection_component(st: Stream) -> Stream:
    """Return one picking trace per station: vertical if present, otherwise highest RMS component."""
    chosen = Stream()
    for sta in stream_station_names(st):
        if not is_fixed_position_station(sta):
            continue
        traces = [tr for tr in st if tr.stats.station == sta]
        if not traces:
            continue
        vertical = [tr for tr in traces if tr.stats.channel.upper().endswith("Z") or tr.stats.channel.upper() in ("DPZ", "EHZ", "BHZ", "HHZ")]
        if vertical:
            chosen += vertical[0]
        else:
            chosen += sorted(traces, key=robust_trace_energy, reverse=True)[0]
    return chosen


def pick_first_arrivals(st_event: Stream) -> pd.DataFrame:
    st_pick = choose_detection_component(preprocess_for_arrivals(st_event))
    rows = []
    if len(st_pick) == 0:
        return pd.DataFrame(rows)
    t_ref = min(tr.stats.starttime for tr in st_pick)

    for tr in st_pick:
        sr = float(tr.stats.sampling_rate)
        nsta = max(1, int(round(STA_S * sr)))
        nlta = max(nsta + 1, int(round(LTA_S * sr)))
        x = np.asarray(tr.data, dtype=float)
        if x.size < nlta * 2 or not np.isfinite(x).any():
            continue
        cft = classic_sta_lta(x, nsta, nlta)
        triggers = trigger_onset(cft, TRIGGER_ON, TRIGGER_OFF)
        if len(triggers) == 0:
            absx = np.abs(x)
            med = np.nanmedian(absx)
            mad = 1.4826 * np.nanmedian(np.abs(absx - med))
            idxs = np.where(absx > med + 8.0 * mad)[0]
            if len(idxs) == 0:
                continue
            i_pick = int(idxs[0])
        else:
            i_pick = int(triggers[0][0])
        pick_time = tr.stats.starttime + i_pick / sr
        rows.append({
            "station": tr.stats.station,
            "channel": tr.stats.channel,
            "pick_time": str(pick_time),
            "pick_time_s": float(pick_time - t_ref),
            "cft_peak": float(np.nanmax(cft)) if len(cft) else np.nan,
            "rms": robust_trace_energy(tr),
            "npts": int(tr.stats.npts),
        })
    df = pd.DataFrame(rows)
    if len(df):
        df = df.sort_values("pick_time_s").reset_index(drop=True)
    return df


def fit_source_position_from_picks(picks: pd.DataFrame, geometry: pd.DataFrame) -> dict:
    if len(picks) < MIN_PICK_COUNT_FOR_FIT:
        return {"success": False, "reason": f"Not enough picks ({len(picks)} < {MIN_PICK_COUNT_FOR_FIT})"}
    df = picks.merge(geometry[["station", "x_m"]], on="station", how="inner").dropna(subset=["x_m", "pick_time_s"])
    if len(df) < MIN_PICK_COUNT_FOR_FIT:
        return {"success": False, "reason": f"Not enough picks with geometry ({len(df)} < {MIN_PICK_COUNT_FOR_FIT})"}

    xvals = df["x_m"].to_numpy(float)
    tvals = df["pick_time_s"].to_numpy(float)
    source_grid = np.arange(np.nanmin(xvals), np.nanmax(xvals) + SOURCE_GRID_DX_M, SOURCE_GRID_DX_M)
    best = None
    for xs in source_grid:
        d = np.abs(xvals - xs)
        for v in VELOCITY_GRID_MPS:
            t0_candidates = tvals - d / v
            t0 = np.nanmedian(t0_candidates)
            pred = t0 + d / v
            resid = tvals - pred
            mad = np.nanmedian(np.abs(resid - np.nanmedian(resid)))
            rmse = float(np.sqrt(np.nanmean(resid**2)))
            score = float(mad if np.isfinite(mad) and mad > 0 else rmse)
            item = {
                "success": True,
                "method": "grid_arrival_fit",
                "source_x_m": float(xs),
                "velocity_mps": float(v),
                "origin_time_s_rel": float(t0),
                "rmse_s": rmse,
                "mad_s": float(mad),
                "score_s": score,
                "n_picks_used": int(len(df)),
            }
            if best is None or item["score_s"] < best["score_s"]:
                best = item
    geom = geometry.copy()
    geom["distance_to_source_m"] = np.abs(geom["x_m"] - best["source_x_m"])
    nearest = geom.sort_values("distance_to_source_m").head(N_NEAREST_NODES)
    best["nearest_stations"] = nearest["station"].tolist()
    best["nearest_station_distances_m"] = nearest["distance_to_source_m"].astype(float).tolist()
    return best


def fallback_source_position_from_earliest_pick(picks: pd.DataFrame, geometry: pd.DataFrame) -> dict:
    if len(picks) == 0:
        return {"success": False, "reason": "No picks available"}
    df = picks.merge(geometry[["station", "x_m"]], on="station", how="inner")
    if len(df) == 0:
        return {"success": False, "reason": "No picks with geometry"}
    row = df.sort_values("pick_time_s").iloc[0]
    source_x = float(row["x_m"])
    geom = geometry.copy()
    geom["distance_to_source_m"] = np.abs(geom["x_m"] - source_x)
    nearest = geom.sort_values("distance_to_source_m").head(N_NEAREST_NODES)
    return {
        "success": True,
        "method": "earliest_pick_fallback",
        "source_x_m": source_x,
        "velocity_mps": np.nan,
        "origin_time_s_rel": float(row["pick_time_s"]),
        "rmse_s": np.nan,
        "mad_s": np.nan,
        "score_s": np.nan,
        "n_picks_used": int(len(df)),
        "nearest_stations": nearest["station"].tolist(),
        "nearest_station_distances_m": nearest["distance_to_source_m"].astype(float).tolist(),
    }


def estimate_source_location(st_event: Stream, geometry: pd.DataFrame) -> tuple[dict, pd.DataFrame]:
    picks = pick_first_arrivals(st_event)
    fit = fit_source_position_from_picks(picks, geometry)
    if not fit.get("success", False):
        fallback = fallback_source_position_from_earliest_pick(picks, geometry)
        if fallback.get("success", False):
            fit = fallback
        else:
            fit["method"] = "failed"
    return fit, picks


## 4. Source waveform extraction, spectra, and SPECFEM2D export

In [4]:
def trim_trace_safely(tr: Trace, t0: UTCDateTime, t1: UTCDateTime) -> Trace | None:
    if t1 <= tr.stats.starttime or t0 >= tr.stats.endtime:
        return None
    tr2 = tr.copy()
    tr2.trim(t0, t1, pad=True, fill_value=0)
    return tr2


def extract_source_waveforms(st_event: Stream, fit: dict, picks: pd.DataFrame) -> dict:
    out = {}
    if not fit.get("success", False):
        return out
    nearest_stations = fit.get("nearest_stations", [])[:N_NEAREST_NODES]
    if not nearest_stations:
        return out

    event_t0 = min(tr.stats.starttime for tr in st_event)
    pick_lookup = {str(r["station"]): UTCDateTime(r["pick_time"]) for _, r in picks.iterrows()} if len(picks) else {}

    for sta in nearest_stations:
        station_traces = Stream([tr for tr in st_event if tr.stats.station == sta])
        if len(station_traces) == 0:
            continue
        if sta in pick_lookup:
            center_time = pick_lookup[sta]
            center_source = "observed_pick"
        else:
            center_time = event_t0 + float(fit.get("origin_time_s_rel", 0.0))
            center_source = "origin_time_fallback"
        t0 = center_time - SOURCE_PRE_S
        t1 = center_time + SOURCE_POST_S
        for tr in station_traces:
            trw = trim_trace_safely(tr, t0, t1)
            if trw is None:
                continue
            trw.detrend("demean")
            trw.detrend("linear")
            trw.taper(max_percentage=SPECTRUM_TAPER_FRACTION)
            key = f"{sta}.{tr.stats.channel}"
            out[key] = {
                "trace": trw,
                "station": sta,
                "channel": tr.stats.channel,
                "window_start": str(t0),
                "window_end": str(t1),
                "center_time": str(center_time),
                "center_source": center_source,
            }
    return out


def spectrum_metrics(tr: Trace) -> tuple[pd.DataFrame, dict]:
    x = np.asarray(tr.data, dtype=float)
    sr = float(tr.stats.sampling_rate)
    n = len(x)
    if n < 2:
        return pd.DataFrame(), {}
    x = x - np.nanmean(x)
    ntaper = int(max(1, round(SPECTRUM_TAPER_FRACTION * n)))
    win = np.ones(n)
    if ntaper > 1 and 2 * ntaper < n:
        taper = 0.5 * (1 - np.cos(np.linspace(0, np.pi, ntaper)))
        win[:ntaper] = taper
        win[-ntaper:] = taper[::-1]
    xw = x * win
    freqs = np.fft.rfftfreq(n, d=1.0 / sr)
    amp = np.abs(np.fft.rfft(xw))
    power = amp**2
    df_spec = pd.DataFrame({"frequency_hz": freqs, "amplitude": amp, "power": power})
    valid = freqs > 0
    if not np.any(valid) or np.nansum(power[valid]) <= 0:
        return df_spec, {"dominant_frequency_hz": np.nan, "centroid_frequency_hz": np.nan, "bandwidth_rms_hz": np.nan, "f05_hz": np.nan, "f95_hz": np.nan}
    f = freqs[valid]
    p = power[valid]
    p_sum = np.nansum(p)
    centroid = float(np.nansum(f * p) / p_sum)
    bw = float(np.sqrt(np.nansum(((f - centroid) ** 2) * p) / p_sum))
    dom = float(f[np.nanargmax(p)])
    cdf = np.cumsum(p) / p_sum
    f05 = float(np.interp(0.05, cdf, f))
    f95 = float(np.interp(0.95, cdf, f))
    return df_spec, {"dominant_frequency_hz": dom, "centroid_frequency_hz": centroid, "bandwidth_rms_hz": bw, "f05_hz": f05, "f95_hz": f95}


def export_specfem_source_time_function(tr: Trace, out_txt: Path, normalize=True) -> dict:
    x = np.asarray(tr.data, dtype=float).copy()
    sr = float(tr.stats.sampling_rate)
    t = np.arange(len(x)) / sr
    x = x - np.nanmean(x)
    maxabs = float(np.nanmax(np.abs(x))) if len(x) else np.nan
    if normalize and np.isfinite(maxabs) and maxabs > 0:
        y = x / maxabs
        normalization = maxabs
    else:
        y = x
        normalization = 1.0
    np.savetxt(out_txt, np.column_stack([t, y]), fmt="%.9e", header="time_s amplitude", comments="# ")
    return {"specfem_source_file": str(out_txt), "sample_rate_hz": sr, "n_samples": int(len(x)), "duration_s": float(t[-1]) if len(t) else 0.0, "normalized": bool(normalize), "normalization_factor": float(normalization)}


def plot_waveform_and_spectrum(tr: Trace, df_spec: pd.DataFrame, title: str, outfile: Path):
    x = np.asarray(tr.data, dtype=float)
    sr = float(tr.stats.sampling_rate)
    t = np.arange(len(x)) / sr
    fig = plt.figure(figsize=(10, 6))
    ax1 = fig.add_subplot(2, 1, 1)
    ax1.plot(t, x, lw=0.8)
    ax1.set_xlabel("Time since source-window start (s)")
    ax1.set_ylabel("Amplitude")
    ax1.set_title(title)
    ax2 = fig.add_subplot(2, 1, 2)
    if len(df_spec):
        ax2.plot(df_spec["frequency_hz"], df_spec["amplitude"], lw=0.8)
        ax2.set_xlim(0, min(250, df_spec["frequency_hz"].max()))
    ax2.set_xlabel("Frequency (Hz)")
    ax2.set_ylabel("FFT amplitude")
    fig.tight_layout()
    fig.savefig(outfile, dpi=150)
    plt.close(fig)


## 5. Process one event

In [12]:
def safe_event_id_from_path(path: Path) -> str:
    return path.stem.replace(" ", "_").replace(":", "").replace("/", "_")


def process_event_mseed(event_path: Path, geometry_unused: pd.DataFrame, outdir: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    event_path = Path(event_path)
    event_id = safe_event_id_from_path(event_path)
    event_out = outdir / event_id
    wf_out = event_out / "specfem_source_time_functions"
    spec_out = event_out / "spectra_csv"
    fig_out = event_out / "figures"
    meta_out = event_out / "metadata"
    for d in [wf_out, spec_out, fig_out, meta_out]:
        d.mkdir(parents=True, exist_ok=True)

    st = read(str(event_path))
    if len(st) == 0:
        raise ValueError(f"No traces read from {event_path}")

    # Exclude moving trigger/source nodes and any non-position-coded traces from geometry-based analysis.
    st = Stream([tr for tr in st if is_fixed_position_station(tr.stats.station)])
    if len(st) == 0:
        raise ValueError(f"No fixed position-coded receiver traces remain after filtering: {event_path}")

    geometry = geometry_from_stream_station_codes(st)
    if len(geometry) == 0:
        raise ValueError(f"Could not derive geometry from station codes in {event_path}")

    geometry_file = meta_out / f"{event_id}_receiver_geometry.csv"
    geometry.to_csv(geometry_file, index=False)

    fit, picks = estimate_source_location(st, geometry)
    source_waveforms = extract_source_waveforms(st, fit, picks)

    picks_file = meta_out / f"{event_id}_arrival_picks.csv"
    picks.to_csv(picks_file, index=False)
    fit_file = meta_out / f"{event_id}_source_location_fit.json"
    with open(fit_file, "w") as f:
        json.dump(fit, f, indent=2)

    waveform_rows = []
    for key, item in source_waveforms.items():
        tr = item["trace"]
        station = item["station"]
        channel = item["channel"]
        base = f"{event_id}_{station}_{channel}"
        spec_df, spec_metrics = spectrum_metrics(tr)
        spec_csv = spec_out / f"{base}_spectrum.csv"
        spec_df.to_csv(spec_csv, index=False)
        specfem_txt = wf_out / f"{base}_specfem_source_time_function.txt"
        export_meta = export_specfem_source_time_function(tr, specfem_txt, normalize=NORMALIZE_EXPORT)
        waveform_mseed = wf_out / f"{base}_source_window.mseed"
        Stream([tr]).write(str(waveform_mseed), format="MSEED")
        fig_file = fig_out / f"{base}_waveform_spectrum.png"
        plot_waveform_and_spectrum(tr, spec_df, title=f"{event_id} {station}.{channel}", outfile=fig_file)
        row = {
            "event_id": event_id,
            "event_mseed": str(event_path),
            "station": station,
            "channel": channel,
            "window_start": item["window_start"],
            "window_end": item["window_end"],
            "center_time": item["center_time"],
            "center_source": item["center_source"],
            "source_x_m": fit.get("source_x_m", np.nan),
            "velocity_mps": fit.get("velocity_mps", np.nan),
            "origin_time_s_rel": fit.get("origin_time_s_rel", np.nan),
            "source_fit_score_s": fit.get("score_s", np.nan),
            "source_fit_rmse_s": fit.get("rmse_s", np.nan),
            "source_fit_mad_s": fit.get("mad_s", np.nan),
            "n_picks_used": fit.get("n_picks_used", np.nan),
            "source_location_method": fit.get("method", ""),
            "source_location_success": fit.get("success", False),
            "waveform_mseed": str(waveform_mseed),
            "spectrum_csv": str(spec_csv),
            "figure_png": str(fig_file),
        }
        row.update(export_meta)
        row.update(spec_metrics)
        waveform_rows.append(row)

    waveform_summary = pd.DataFrame(waveform_rows)
    event_summary = pd.DataFrame([{
        "event_id": event_id,
        "event_mseed": str(event_path),
        "n_traces": len(st),
        "event_starttime": str(min(tr.stats.starttime for tr in st)),
        "event_endtime": str(max(tr.stats.endtime for tr in st)),
        "source_location_success": fit.get("success", False),
        "source_location_method": fit.get("method", ""),
        "source_x_m": fit.get("source_x_m", np.nan),
        "velocity_mps": fit.get("velocity_mps", np.nan),
        "origin_time_s_rel": fit.get("origin_time_s_rel", np.nan),
        "source_fit_score_s": fit.get("score_s", np.nan),
        "source_fit_rmse_s": fit.get("rmse_s", np.nan),
        "source_fit_mad_s": fit.get("mad_s", np.nan),
        "n_picks_used": fit.get("n_picks_used", np.nan),
        "nearest_stations": ",".join(fit.get("nearest_stations", [])),
        "picks_csv": str(picks_file),
        "source_fit_json": str(fit_file),
        "geometry_csv": str(geometry_file),
        "event_output_dir": str(event_out),
        "n_source_waveforms_exported": len(waveform_summary),
    }])
    return event_summary, waveform_summary, picks


## 6. Select event files

In [13]:
from pathlib import Path

def find_event_files(root: Path, pattern: str = "*.mseed") -> list[Path]:
    root = Path(root)

    if root.is_file():
        if root.name.startswith("._"):
            return []
        return [root]

    return sorted(
        f for f in root.rglob(pattern)
        if (
            f.is_file()
            and not f.name.startswith("._")
            and f.stat().st_size > 0
        )
    )

if RUN_MODE == "single":
    if SINGLE_EVENT_MSEED is None:
        raise ValueError("Set SINGLE_EVENT_MSEED when RUN_MODE == 'single'")
    event_files = find_event_files(Path(SINGLE_EVENT_MSEED))
else:
    event_files = find_event_files(BATCH_SEARCH_ROOT, EVENT_FILE_GLOB)

print(f"Found {len(event_files)} event files")
for p in event_files[:10]:
    print(p)


Found 2219 event files
/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_position_codes_timewindows/T1_N1_Streamer/T1_N1_DPall/events_mseed/T1_N1_Streamer_T1_N1_E00001_000.mseed
/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_position_codes_timewindows/T1_N1_Streamer/T1_N1_DPall/events_mseed/T1_N1_Streamer_T1_N1_E00002_000.mseed
/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_position_codes_timewindows/T1_N1_Streamer/T1_N1_DPall/events_mseed/T1_N1_Streamer_T1_N1_E00003_000.mseed
/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_position_codes_timewindows/T1_N1_Streamer/T1_N1_DPall/events_mseed/T1_N1_Streamer_T1_N1_E00004_000.mseed
/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_position_codes_timewindows/T1_N1_Streamer/T1_N1_DPall/events_mseed/T1_N1_Streamer_T1_N1_E00005_000.mseed
/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_position_codes_timewindows/T1_N1_Streamer/T1_N1_DPall/events_mseed/T1_N1_Streamer_T1_N1_E00006_000.mseed
/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_position_codes_timewindow

## 7. Run extraction

For testing, keep `MAX_EVENTS_TO_PROCESS = 1`. Set it to `None` once the result looks sensible.

In [14]:
MAX_EVENTS_TO_PROCESS = 1  # set to None for all

# Geometry is now derived inside process_event_mseed() from the position-coded
# station names in each event file, so there is no global geometry table to check here.
if not event_files:
    raise RuntimeError(f"No event MiniSEED files found under {BATCH_SEARCH_ROOT} matching {EVENT_FILE_GLOB}")

files_to_process = event_files if MAX_EVENTS_TO_PROCESS is None else event_files[:MAX_EVENTS_TO_PROCESS]
all_event_summaries = []
all_waveform_summaries = []
failed = []

for i, event_path in enumerate(files_to_process, start=1):
    print(f"[{i}/{len(files_to_process)}] {event_path}")
    try:
        event_summary, waveform_summary, picks = process_event_mseed(event_path, geometry, OUTDIR)
        all_event_summaries.append(event_summary)
        all_waveform_summaries.append(waveform_summary)
        print("  source_x_m:", event_summary.iloc[0]["source_x_m"])
        print("  nearest:", event_summary.iloc[0]["nearest_stations"])
        print("  exported waveforms:", event_summary.iloc[0]["n_source_waveforms_exported"])
        display(event_summary)
        if len(waveform_summary):
            display(waveform_summary.head())
    except Exception as e:
        print("  FAILED:", e)
        failed.append({"event_mseed": str(event_path), "error": repr(e)})

df_event_summary = pd.concat(all_event_summaries, ignore_index=True) if all_event_summaries else pd.DataFrame()
df_waveform_summary = pd.concat(all_waveform_summaries, ignore_index=True) if all_waveform_summaries else pd.DataFrame()
df_failed = pd.DataFrame(failed)
summary_dir = OUTDIR / "_summary_tables"
summary_dir.mkdir(parents=True, exist_ok=True)
df_event_summary.to_csv(summary_dir / "source_waveform_event_summary.csv", index=False)
df_waveform_summary.to_csv(summary_dir / "source_waveform_exports.csv", index=False)
df_failed.to_csv(summary_dir / "source_waveform_failures.csv", index=False)
print("Summary tables written to:", summary_dir)
print("Events processed:", len(df_event_summary))
print("Waveforms exported:", len(df_waveform_summary))
print("Failures:", len(df_failed))


[1/1] /Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_position_codes_timewindows/T1_N1_Streamer/T1_N1_DPall/events_mseed/T1_N1_Streamer_T1_N1_E00001_000.mseed
  source_x_m: 91.58
  nearest: 09210,08808
  exported waveforms: 6


,event_id,event_mseed,n_traces,event_starttime,event_endtime,source_location_success,source_location_method,source_x_m,velocity_mps,origin_time_s_rel,source_fit_score_s,source_fit_rmse_s,source_fit_mad_s,n_picks_used,nearest_stations,picks_csv,source_fit_json,geometry_csv,event_output_dir,n_source_waveforms_exported
0,T1_N1_Streamer_T1_N1_E00001_000,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...,18,2026-05-16T17:12:15.508000Z,2026-05-16T17:12:16.258000Z,True,grid_arrival_fit,91.58,700.0,0.047129,0.000743,0.00627,0.000743,6,"09210,08808",/Volumes/tachyon/LBSSP_DATA/nodal_source_wavef...,/Volumes/tachyon/LBSSP_DATA/nodal_source_wavef...,/Volumes/tachyon/LBSSP_DATA/nodal_source_wavef...,/Volumes/tachyon/LBSSP_DATA/nodal_source_wavef...,6


,event_id,event_mseed,station,channel,window_start,window_end,center_time,center_source,source_x_m,velocity_mps,...,sample_rate_hz,n_samples,duration_s,normalized,normalization_factor,dominant_frequency_hz,centroid_frequency_hz,bandwidth_rms_hz,f05_hz,f95_hz
0,T1_N1_Streamer_T1_N1_E00001_000,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...,09210,DPE,2026-05-16T17:12:15.526000Z,2026-05-16T17:12:16.056000Z,2026-05-16T17:12:15.556000Z,observed_pick,91.58,700.0,...,500.0,266,0.53,True,0.818285,24.436090,38.678708,19.002962,16.976516,75.851111
1,T1_N1_Streamer_T1_N1_E00001_000,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...,09210,DPN,2026-05-16T17:12:15.526000Z,2026-05-16T17:12:16.056000Z,2026-05-16T17:12:15.556000Z,observed_pick,91.58,700.0,...,500.0,266,0.53,True,0.972511,20.676692,30.230645,13.910994,17.442329,50.980500
2,T1_N1_Streamer_T1_N1_E00001_000,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...,09210,DPZ,2026-05-16T17:12:15.526000Z,2026-05-16T17:12:16.056000Z,2026-05-16T17:12:15.556000Z,observed_pick,91.58,700.0,...,500.0,266,0.53,True,0.819526,24.436090,28.970075,10.903315,18.776372,53.454389
3,T1_N1_Streamer_T1_N1_E00001_000,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...,08808,DPE,2026-05-16T17:12:15.530000Z,2026-05-16T17:12:16.060000Z,2026-05-16T17:12:15.560000Z,observed_pick,91.58,700.0,...,500.0,266,0.53,True,0.811268,20.676692,49.785088,29.073147,17.253962,93.444062
4,T1_N1_Streamer_T1_N1_E00001_000,/Volumes/tachyon/LBSSP_DATA/nodal_shotgathers_...,08808,DPN,2026-05-16T17:12:15.530000Z,2026-05-16T17:12:16.060000Z,2026-05-16T17:12:15.560000Z,observed_pick,91.58,700.0,...,500.0,266,0.53,True,1.068874,18.796992,32.621668,17.259170,17.360810,71.386874


Summary tables written to: /Volumes/tachyon/LBSSP_DATA/nodal_source_waveforms_for_specfem2d/_summary_tables
Events processed: 1
Waveforms exported: 6
Failures: 0


## 8. Quick review

In [8]:
if len(df_waveform_summary):
    cols = [
        "event_id", "station", "channel", "source_x_m", "velocity_mps",
        "dominant_frequency_hz", "centroid_frequency_hz", "f05_hz", "f95_hz",
        "specfem_source_file", "figure_png"
    ]
    display(df_waveform_summary[[c for c in cols if c in df_waveform_summary.columns]].head(20))
else:
    print("No waveform summary rows yet.")


No waveform summary rows yet.


## 9. SPECFEM2D notes

The exported files named `*_specfem_source_time_function.txt` contain:

```text
time_s amplitude
```

By default they are normalized to peak absolute amplitude = 1. The normalization factor is stored in `_summary_tables/source_waveform_exports.csv`.

Recommended refinements:

- use existing first-break picks if available instead of STA/LTA repicking,
- constrain source position from the Geode/field-note source catalog,
- separate PEG, sledgehammer, and Betsy source waveforms,
- export one preferred component if SPECFEM2D requires a scalar source,
- stack repeated shots at the same source location to estimate a more stable empirical source wavelet.
